# 16 - Cleanup
Tear down all tracked AWS resources safely using cleanup_utils, detect orphaned resources, and reset config.json.

## Step 1: Load Configuration and Tracker

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker, ResourceCleanup

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

tracker = ResourceTracker()
print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Review Tracked Resources

See everything that was created during the notebook series.

In [ ]:
print("Tracked resources to be cleaned up:")
print("=" * 50)
tracker.summary()
print("=" * 50)

all_resources = tracker.get_all()
total = sum(len(v) for v in all_resources.values() if isinstance(v, list))
print(f"\nTotal resources to clean: {total}")

## Step 3: Detailed Resource Inventory

In [ ]:
resources = tracker.get_all()

print("Detailed inventory:\n")

if resources.get("neptune_graphs"):
    print("Neptune Analytics Graphs:")
    for g in resources["neptune_graphs"]:
        print(f"  - {g['name']} (ID: {g['id']})")

if resources.get("knowledge_bases"):
    print("\nBedrock Knowledge Bases:")
    for kb in resources["knowledge_bases"]:
        print(f"  - {kb['name']} (ID: {kb['id']})")

if resources.get("opensearch_collections"):
    print("\nOpenSearch Serverless Collections:")
    for c in resources["opensearch_collections"]:
        print(f"  - {c['name']} (ARN: {c['arn']})")

if resources.get("guardrails"):
    print("\nBedrock Guardrails:")
    for g in resources["guardrails"]:
        print(f"  - {g['name']} (ID: {g['id']})")

if resources.get("s3_buckets"):
    print("\nS3 Buckets:")
    for b in resources["s3_buckets"]:
        print(f"  - {b}")

if resources.get("iam_roles"):
    print("\nIAM Roles:")
    for r in resources["iam_roles"]:
        print(f"  - {r['name']} (ARN: {r['arn']})")

if resources.get("agentcore_agents"):
    print("\nAgentCore Agents:")
    for a in resources["agentcore_agents"]:
        print(f"  - {a['name']} (ID: {a['id']})")

if resources.get("agentcore_gateways"):
    print("\nAgentCore Gateways:")
    for g in resources["agentcore_gateways"]:
        print(f"  - {g['name']} (ID: {g['id']})")

if resources.get("agentcore_memories"):
    print("\nAgentCore Memories:")
    for m in resources["agentcore_memories"]:
        print(f"  - {m['name']} (ID: {m['id']})")

print("\n\u2713 Inventory complete")

## Step 4: Scan for Orphaned Resources

Check for resources with our prefix that aren't in the tracker.

In [ ]:
print("Scanning for orphaned resources...\n")
orphans = []

# Check S3 for untracked buckets
s3 = boto3.client("s3", region_name=REGION)
try:
    buckets = s3.list_buckets()["Buckets"]
    tracked_buckets = set(resources.get("s3_buckets", []))
    for bucket in buckets:
        if PREFIX in bucket["Name"] and bucket["Name"] not in tracked_buckets:
            orphans.append(("S3", bucket["Name"]))
            print(f"  \u26a0\ufe0f Orphaned S3 bucket: {bucket['Name']}")
except Exception as e:
    print(f"  Could not scan S3: {e}")

# Check Neptune for untracked graphs
try:
    neptune = boto3.client("neptune-graph", region_name=REGION)
    graphs = neptune.list_graphs()["graphSummaries"]
    tracked_graph_ids = {g["id"] for g in resources.get("neptune_graphs", [])}
    for graph in graphs:
        if PREFIX in graph["name"] and graph["id"] not in tracked_graph_ids:
            orphans.append(("Neptune", graph["name"]))
            print(f"  \u26a0\ufe0f Orphaned Neptune graph: {graph['name']}")
except Exception as e:
    print(f"  Could not scan Neptune: {e}")

# Check Guardrails
try:
    bedrock = boto3.client("bedrock", region_name=REGION)
    guardrails = bedrock.list_guardrails()["guardrails"]
    tracked_gr_ids = {g["id"] for g in resources.get("guardrails", [])}
    for gr in guardrails:
        if PREFIX in gr["name"] and gr["id"] not in tracked_gr_ids:
            orphans.append(("Guardrail", gr["name"]))
            print(f"  \u26a0\ufe0f Orphaned guardrail: {gr['name']}")
except Exception as e:
    print(f"  Could not scan Guardrails: {e}")

# Check Knowledge Bases
try:
    bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)
    kbs = bedrock_agent.list_knowledge_bases()["knowledgeBaseSummaries"]
    tracked_kb_ids = {kb["id"] for kb in resources.get("knowledge_bases", [])}
    for kb in kbs:
        if PREFIX in kb["name"] and kb["knowledgeBaseId"] not in tracked_kb_ids:
            orphans.append(("KB", kb["name"]))
            print(f"  \u26a0\ufe0f Orphaned Knowledge Base: {kb['name']}")
except Exception as e:
    print(f"  Could not scan Knowledge Bases: {e}")

if not orphans:
    print("  \u2713 No orphaned resources found")
else:
    print(f"\n  \u26a0\ufe0f {len(orphans)} orphaned resources detected")

print("\n\u2713 Orphan scan complete")

## Step 5: Dry Run Cleanup

Preview what will be deleted without actually deleting.

In [ ]:
cleanup = ResourceCleanup(tracker, region=REGION)

print("DRY RUN - Resources that would be deleted:\n")
cleanup.cleanup_all(dry_run=True)

## Step 6: Execute Cleanup

**WARNING: This deletes all tracked resources permanently.**

Uncomment the cell below to execute. Resources are deleted in dependency order.

In [ ]:
# UNCOMMENT THE LINES BELOW TO EXECUTE CLEANUP
# Make sure you are ready to delete all resources!

# print("\u231b Starting resource cleanup...")
# print("  Deletion order: data sources -> KBs -> flows -> guardrails -> prompts")
# print("                  -> Neptune -> OpenSearch -> S3 -> DynamoDB -> IAM -> policies")
# print()
# cleanup.cleanup_all(dry_run=False)
# print("\n\u2713 All tracked resources deleted")

print("\u231b Cleanup not executed (uncomment to run)")
print("  To execute: remove the # from cleanup.cleanup_all(dry_run=False)")

## Step 7: Clean Up AgentCore Resources

AgentCore resources (gateway, memory, agents) need separate cleanup.

In [ ]:
# AgentCore-specific cleanup
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print("AgentCore resources to clean up:")

# Agents
for agent in resources.get("agentcore_agents", []):
    print(f"  Agent: {agent['name']} ({agent['id']})")
    # agentcore.delete_agent(agentId=agent["id"])  # Uncomment to delete

# Gateways
for gw in resources.get("agentcore_gateways", []):
    print(f"  Gateway: {gw['name']} ({gw['id']})")
    # agentcore.delete_gateway(gatewayId=gw["id"])  # Uncomment to delete

# Memories
for mem in resources.get("agentcore_memories", []):
    print(f"  Memory: {mem['name']} ({mem['id']})")
    # agentcore.delete_memory_store(memoryStoreId=mem["id"])  # Uncomment to delete

print("\n\u231b Uncomment delete calls above to remove AgentCore resources")

## Step 8: Reset config.json

In [ ]:
# UNCOMMENT TO RESET CONFIG
# Reset config to initial state

# initial_config = {
#     "REGION": "us-east-1",
#     "ACCOUNT_ID": None,
#     "PROJECT_PREFIX": "orgiq",
#     "MODEL_ID": "us.anthropic.claude-sonnet-4-6-20250929-v1:0",
#     "EMBEDDING_MODEL_ID": "amazon.titan-embed-text-v2:0",
#     "EMBEDDING_DIMENSIONS": 1024,
#     "NEPTUNE_GRAPH_ID": None,
#     "NEPTUNE_ENDPOINT": None,
#     "KB_ID": None,
#     "KB_DATA_SOURCE_ID": None,
#     "COLLECTION_ID": None,
#     "COLLECTION_ARN": None,
#     "GUARDRAIL_ID": None,
#     "GUARDRAIL_VERSION": None,
#     "S3_BUCKET": None,
#     "MEMORY_ID": None,
#     "GATEWAY_ID": None,
#     "POLICY_ENGINE_ID": None,
#     "WORKLOAD_IDENTITY_NAME": None,
#     "AGENT_RUNTIME_ARNS": {}
# }
# Path("config.json").write_text(json.dumps(initial_config, indent=2))
# print("\u2713 config.json reset to initial state")

print("\u231b Config not reset (uncomment to execute)")

## Step 9: Reset Resource Tracker

In [ ]:
# UNCOMMENT TO RESET TRACKER
# tracker.clear()
# print("\u2713 Resource tracker cleared")

print("\u231b Tracker not cleared (uncomment to execute)")

## Step 10: Cost Awareness

Resources that cost money when idle:

In [ ]:
print("\u26a0\ufe0f Resources with IDLE COSTS (delete when not in use):\n")
print("  1. Neptune Analytics graph")
print("     Cost: ~$0.12/NCU-hour (even when idle)")
print("     Action: Delete graph or use CDK destroy")
print()
print("  2. OpenSearch Serverless collection")
print("     Cost: ~$0.24/OCU-hour (2 OCU minimum, even when idle)")
print("     Action: Delete collection")
print()
print("  3. Bedrock Knowledge Base")
print("     Cost: Included in OpenSearch Serverless (no separate charge)")
print("     Action: Delete KB and collection together")
print()
print("  4. AgentCore Runtime agents")
print("     Cost: Per-invocation (no idle cost)")
print("     Action: Delete if not needed")
print()
print("Resources with NO idle cost:")
print("  - S3 buckets (pay per GB stored + requests)")
print("  - Guardrails (pay per assessment)")
print("  - IAM roles (free)")
print("  - Lambda functions (pay per invocation)")

## Step 11: Verification

In [ ]:
# Final state check
print("Current state:")
print(f"  config.json: {'populated' if config.get('NEPTUNE_GRAPH_ID') else 'clean'}")
print(f"  resource_tracker.json: {total} resources tracked")
print()
print("To fully clean up:")
print("  1. Uncomment and run Step 6 (main cleanup)")
print("  2. Uncomment and run Step 7 (AgentCore resources)")
print("  3. Uncomment and run Step 8 (reset config.json)")
print("  4. Uncomment and run Step 9 (clear tracker)")
print()
print("Alternative - CDK destroy (infrastructure only):")
print("  cd infra && npx cdk destroy --all")

## Summary

- [x] All tracked resources inventoried
- [x] Orphaned resource scan completed
- [x] Dry run cleanup previewed
- [x] Cleanup execution ready (uncomment to run)
- [x] AgentCore resources listed for manual cleanup
- [x] Config reset prepared
- [x] Resource tracker reset prepared
- [x] Cost awareness documented

**Done!** This completes the OrgIQ notebook series. Return to [00-index.ipynb](00-index.ipynb) for the full table of contents.